In [ ]:
!pip install torch transformers

In [ ]:
import json
import os
import torch
CLASSES = {
    "Adware": 0,
    "Backdoor": 1,
    "Botnet": 2,
    "CGI": 3,
    "Code-execution": 4,
    "DDos": 5,
    "Dir-Traversal": 6,
    "Dos": 7,
    "Info-Disclosure": 8,
    "Injection": 9,
    "Other": 10,
    "Overflow": 11,
    "Ransomware": 12,
    "Remote-file-Inclusion": 13,
    "Scanner": 14,
    "Spyware": 15,
    "Trojan": 16,
    "Virus": 17,
    "Webshell": 18,
    "Worm": 19,
    "XSS": 20
}
INV_CLASSES = {v: k for k, v in CLASSES.items()}
CONCEPTS= ["ip", "injection"]
CLASSES_TO_EXAMINE = ["Adware", "Scanner", "Spyware", "Trojan", "XSS", "Remote-file-Inclusion", "Overflow", "Injection", "Info-Disclosure", "Dir-Traversal", "Code-execution", "CGI", "Ransomware", "Botnet", "Backdoor"]
MODEL_NAME = "./codebert-base-mlm"


from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, output_hidden_states=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
print(model)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data as data
import numpy as np
import random

In [ ]:
# Estrazione delle feature
f = open("./data/packet_inspection/packets_dataset.jsonl", "r")
ai_dataset = [json.loads(line) for line in f.readlines()]
f.close()

f = open("./data/packet_inspection/anomalous_packets.jsonl", "r")
val_test_dataset = [json.loads(line) for line in f.readlines()]
f.close()

train_dataset = random.sample(val_test_dataset, 1000)

val_dataset = random.sample([v for v in val_test_dataset if v not in train_dataset], 100)
test_dataset = ai_dataset

# Tokenize each sample individually to avoid padding
tokenized_inputs_list = []
for s in train_dataset:
    tokenized = tokenizer(s["text"], padding=False, truncation=True, return_tensors="pt")
    tokenized_inputs_list.append(tokenized)

# Calcola activations e scales usando tokenized_inputs_list (ogni elemento è un singolo sample tokenizzato)
layer_reshapeds = None
with torch.no_grad():
    for tokenized in tokenized_inputs_list:
        outputs = model(**tokenized)
        hidden_states = outputs.hidden_states  # tuple of tensors (1, seq_len, d)
        if layer_reshapeds is None:
            layer_reshapeds = [[] for _ in range(len(hidden_states))]
        for idx, h in enumerate(hidden_states):
            b, s, d = h.shape  # b == 1
            reshaped = h.reshape(b * s, d)  # (tokens, d)
            layer_reshapeds[idx].append(reshaped)

scales = []
activations = []
# Concatenate per layer, calcola scale e normalizza
for h_list in layer_reshapeds:
    cat = torch.cat(h_list, dim=0)  # (total_tokens_across_samples, d)
    d = cat.shape[1]
    mean_squared_norm = torch.mean(torch.sum(cat ** 2, dim=1))
    scale_t = torch.sqrt(torch.tensor(d, dtype=mean_squared_norm.dtype) / mean_squared_norm)
    scale = float(scale_t.item())
    scales.append(scale)
    normalized = cat * scale
    activations.append(normalized.numpy())
print("Done training inputs")

# Val and test activation calculation adapted to match `activations` (per-sample, no padding)
val_activations = []
val_tokenized_inputs_list = [tokenizer(s["text"], padding=False, truncation=True, return_tensors="pt") for s in val_dataset]

layer_reshapeds_val = None
with torch.no_grad():
    for tokenized in val_tokenized_inputs_list:
        outputs = model(**tokenized)
        hidden_states = outputs.hidden_states
        if layer_reshapeds_val is None:
            layer_reshapeds_val = [[] for _ in range(len(hidden_states))]
        for idx, h in enumerate(hidden_states):
            b, s, d = h.shape
            reshaped = h.reshape(b * s, d)
            layer_reshapeds_val[idx].append(reshaped)

for idx, h_list in enumerate(layer_reshapeds_val):
    cat = torch.cat(h_list, dim=0)
    scale = scales[idx]
    normalized = cat * scale
    val_activations.append(normalized.numpy())
print("Done validating inputs")

test_activations = []
test_tokenized_inputs_list = [tokenizer(s["text"], padding=False, truncation=True, return_tensors="pt") for s in test_dataset]

layer_reshapeds_test = None
with torch.no_grad():
    for tokenized in test_tokenized_inputs_list:
        outputs = model(**tokenized)
        hidden_states = outputs.hidden_states
        if layer_reshapeds_test is None:
            layer_reshapeds_test = [[] for _ in range(len(hidden_states))]
        for idx, h in enumerate(hidden_states):
            b, s, d = h.shape
            reshaped = h.reshape(b * s, d)
            layer_reshapeds_test[idx].append(reshaped)

for idx, h_list in enumerate(layer_reshapeds_test):
    cat = torch.cat(h_list, dim=0)
    scale = scales[idx]
    normalized = cat * scale
    test_activations.append(normalized.numpy())
print("Done testing inputs")

for l in range(len(activations)):
    random.shuffle(activations[l])
    random.shuffle(val_activations[l])
    random.shuffle(test_activations[l])

In [ ]:
class SAE(nn.Module):
    input_dim: int
    hidden_dim: int
    
    def __init__(self, input_dim, hidden_dim, init_weights=True, l2_norm=0.1):
        
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        super(SAE, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim, input_dim),
        )
        if init_weights:
            self.init_weights(l2_norm=l2_norm)
        return
        
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded, encoded
    
    def feature_activations(self, encoded):
        norm_weight = torch.linalg.norm(self.decoder[0].weight, dim=0)
        return encoded * norm_weight
        
    def feature_directions(self):
        norm_weight = torch.linalg.norm(self.decoder[0].weight, dim=0)
        return self.decoder[0].weight / norm_weight
    
    def loss_function(self, recon_x, x, encoded, sparsity_weight):
        return torch.mean(F.mse_loss(recon_x, x) + sparsity_weight * self.feature_activations(encoded))

    def init_weights(self, l2_norm=0.1):
        # Crea una matrice random di shape (input_dim, hidden_dim)
        W_d = torch.randn_like(self.decoder[0].weight)
        # Normalizza ogni colonna a norma L2 = l2_norm
        W_d = W_d / W_d.norm(dim=0, p=2, keepdim=True) * l2_norm
        
        self.decoder[0].weight.copy_(W_d)
        self.encoder[0].weight.copy_(W_d.T)
        self.decoder[0].bias.zero_()
        self.encoder[0].bias.zero_()

        return
    

In [ ]:
from itertools import product
from tqdm import tqdm

SKIP = False
os.makedirs("saved_models", exist_ok=True)

input_dim = 768  # Dimension of RoBERTa embeddings
layers = [6,8,10,12]       # Fissa il layer da usare
beta = 5.0       # Peso della penalizzazione
lr = 5e-5     # Fissa il learning rate
hidden_dims = [768, 768*2, 768*4, 768*8, 768*16, 768*32]  # Diverse dimensioni da provare

if not SKIP:
    previous_results = []
    try:
        with open("./saved_models/training_results.json", "r") as f:
            for l in f.readlines():
                previous_results.append(json.loads(l))
    except FileNotFoundError:
        pass
    results = []

    for layer, hidden_dim in product(layers, hidden_dims):
        # Prepara DataLoader
        X_train = activations[layer]
        train_dataset = data.TensorDataset(torch.from_numpy(X_train))
        train_loader = data.DataLoader(train_dataset, batch_size=32, shuffle=True)
        
        sae = SAE(input_dim, hidden_dim)
        
        optimizer = optim.Adam(sae.parameters(), lr=lr)
        epochs = 200

        # Early stopping parameters
        early_stop_patience = 5
        best_val_loss = float('inf')
        epochs_no_improve = 0
        avg_loss = float('inf')
        # Early stopping training loop
        for epoch in tqdm(range(epochs), desc=f"Training SAE Layer {layer} Hidden {hidden_dim}", postfix={"loss": avg_loss}):
            # Imposta beta: parte basso, cresce linearmente fino a beta target dopo il 10% delle epoche
            if epoch < int(epochs * 0.1):
                curr_beta = beta * (epoch / (epochs * 0.1))
            else:
                curr_beta = beta

            epoch_loss = 0
            sae.train()
            for batch_idx, (data_batch,) in enumerate(train_loader):
                optimizer.zero_grad()
                
                reconstructed_data, encoded_activations = sae(data_batch)
                
                loss = model.loss_function(reconstructed_data, data_batch, encoded_activations, curr_beta)
                loss.backward()
                
                optimizer.step()
                epoch_loss += loss.item()

            avg_loss = epoch_loss / len(X_train)

            # Validazione
            val_X = val_activations[layer]
            val_dataset = data.TensorDataset(torch.from_numpy(val_X))
            val_loader = data.DataLoader(val_dataset, batch_size=32, shuffle=False)

            # Validation loss for early stopping
            sae.eval()
            val_loss = 0
            with torch.no_grad():
                for val_batch, in val_loader:
                    reconstructed_data, encoded_activations = sae(val_batch)
                    v_loss = model.loss_function(reconstructed_data, val_batch, encoded_activations, curr_beta)
                    val_loss += v_loss.item()
                avg_val_loss = val_loss / len(val_X)
                

            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                epochs_no_improve = 0
                # Optionally save best model weights here
                best_model_state = sae.state_dict()
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= early_stop_patience:
                    print(f"Early stopping at epoch {epoch+1}")
                    # Restore best model weights
                    sae.load_state_dict(best_model_state)
                    break
            print(f"hidden_dim={hidden_dim}, beta={curr_beta}, Training_loss={avg_loss:.4f}, Validation_loss={avg_val_loss:.4f}")
        print(f"Final training loss: {avg_loss:.4f}, Best validation loss: {best_val_loss:.4f}")

        # Test
        test_X = test_activations[layer]
        test_dataset = data.TensorDataset(torch.from_numpy(test_X))
        test_loader = data.DataLoader(test_dataset, batch_size=32, shuffle=False)

        test_loss = 0
        with torch.no_grad():
            for test_batch, in test_loader:
                reconstructed_data, encoded_activations = sae(test_batch)
                t_loss = model.loss_function(reconstructed_data, test_batch, encoded_activations, beta)
                test_loss += t_loss.item()
            avg_test_loss = test_loss / len(test_X)
        print(f"Test loss: {avg_test_loss:.4f}")
        
        # Salva ogni risultato
        x = {"layer": layer, "hidden_dim": hidden_dim, "beta": beta, "lr": lr, "avg_loss": avg_loss}
        with open("saved_models/training_results.json", "a") as f:
            json.dump(x, f)
            f.write("\n")
        results.append(x)

        # Salva il modello per ogni combinazione
        torch.save(sae.state_dict(), f"saved_models/sae_layer_{layer}_hiddim_{hidden_dim}.pt")

    print("Tutti i modelli e dettagli di training salvati in 'saved_models/' e 'training_results.json'")


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, output_hidden_states=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
validations = random.sample([d for d in val_test_dataset if d not in train_dataset], 10)
inputs = [s["text"] for s in validations]
tokens_id = tokenizer(inputs, padding=True, truncation=True, return_tensors="pt")
activations =  model(**tokens_id).hidden_states

num_samples, num_tokens, _ = activations[layer].shape
tokens_str = [tokenizer.tokenize(s, truncation=True,padding="max_length", max_length=num_tokens) for s in inputs]

print("Done gathering inputs")


In [ ]:
import umap

# Analyzing feature directions for all saes

hidden_dims = [768, 768*2, 768*4, 768*8, 768*16, 768*32] 
layer = 10
beta = 5.0
lr = 5e-5

feature_directions_dict = {}

for hidden_dim in hidden_dims:
    sae = SAE(input_dim, hidden_dim)
    sae.load_state_dict(torch.load(f"saved_models/sae_layer_{layer}_hiddim_{hidden_dim}.pt"))
    encoder_weights = sae.encoder[0].weight
    decoder_weights = sae.decoder[0].weight
    feature_directions = get_feature_directions(decoder_weights)
    feature_directions_dict[hidden_dim] = feature_directions

    test_X_tensor = activations[layer]
    with torch.no_grad():
        sae.eval()
        _ , feature_activations = sae(test_X_tensor)

    nonzero_counts = (feature_activations != 0).sum(dim=2)  # shape: (100, 512)
    flattened_nonzero_counts = nonzero_counts.flatten()
    sparsity = flattened_nonzero_counts.float().mean().item()
    print(f"[Dim {hidden_dim}] Sparsità media (numero di feature attive per singolo token): {sparsity:.4f}")

    mean_activations = feature_activations.mean(dim=(0,1)) # shape: (hidden_dim)
    live_features = (mean_activations!=0).float().mean().item()
    print(f"[Dim {hidden_dim}] Valore medio feature attive: {live_features:.4f}")

    # Calcola per ogni feature (sull'ultima dimensione) se è sempre zero su tutti i token di tutti i sample
    # feature_activations: (num_samples, num_tokens, hidden_dim)
    # dead_features_mask: (hidden_dim,) True se la feature è sempre zero
    print(feature_activations)
    dead_features = (feature_activations == 0).all(dim=(0, 1)).sum().item()
    print(f"[Dim {hidden_dim}] Numero di dead features (mai attivate): {dead_features} su {len(mean_activations)}")

    top_features = torch.topk(mean_activations, k=10)
    print(f"[Dim {hidden_dim}] Top 10 feature (concetti) più attive:")
    for idx, value in zip(top_features.indices.tolist(), top_features.values.tolist()):
        print(idx, value)
        print(f"Feature {idx}: attivazione media = {value:.4f}")
        

import matplotlib.pyplot as plt

reducer = umap.UMAP(n_components=2, random_state=42)
colors = ['red', 'orange', 'green', 'blue', 'purple', 'black']
plt.figure(figsize=(10, 8))

min_hd = min(hidden_dims)
base_size = 20
# scala la dimensione dei punti in modo decrescente al crescere di hidden_dim
sizes = [max(5, base_size * (min_hd / hd) ** 0.5) for hd in hidden_dims]

for i, hidden_dim in enumerate(hidden_dims):
    fd = feature_directions_dict[hidden_dim].detach().cpu().numpy().T  # shape: (hidden_dim, input_dim)
    embedding = reducer.fit_transform(fd)
    plt.scatter(
        embedding[:, 0], embedding[:, 1],
        label=f"hidden_dim={hidden_dim}",
        alpha=0.6, s=sizes[i], color=colors[i % len(colors)]
    )

plt.title("UMAP 2D delle feature_directions per diversi hidden_dim")
plt.legend(title="Hidden dim")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.show()

In [ ]:
hidden_dim = 768*8
sae = SAE(input_dim, hidden_dim)
sae.load_state_dict(torch.load(f"saved_models/sae_layer_{layer}_hiddim_{hidden_dim}.pt"))
encoder_weights = sae.encoder[0].weight
decoder_weights = sae.decoder[0].weight
feature_directions = get_feature_directions(decoder_weights)
_, feature_activations = sae.eval()(torch.from_numpy(activations[layer]))

token_to_feature_activations = {}
num_samples, num_tokens, _ = activations[layer].shape
# print(activations[layer].shape)
# print(len(tokens_str[0]))
for sample_idx in range(num_samples):
    for token_idx in range(num_tokens):
        if token_idx == 0:
            token_to_feature_activations[(sample_idx, "CLS", token_idx)] = feature_activations[sample_idx, token_idx].detach().cpu().numpy()
        else:
            if tokens_id.attention_mask[sample_idx][token_idx] == 1:
                if tokens_str[sample_idx][token_idx-1] == "<pad>":
                    token_to_feature_activations[(sample_idx, "</s>", token_idx)] = feature_activations[sample_idx, token_idx].detach().cpu().numpy()
                else:
                    token_to_feature_activations[(sample_idx, tokens_str[sample_idx][token_idx-1], token_idx)] = feature_activations[sample_idx, token_idx].detach().cpu().numpy()
            else:
                continue

# print(token_to_feature_activations.keys())
first_key = next(iter(token_to_feature_activations))
print(len(token_to_feature_activations[first_key]))

In [ ]:
from collections import defaultdict
from tqdm import tqdm
# Crea un dizionario che, per ogni feature, raccoglie i token stringa che hanno attivazione non zero per quella feature
# Ora includi anche il token_idx nella tupla

feature_to_tokens = defaultdict(list)  # feature_idx -> lista di tuple (token stringa, sample_idx, token_idx, activation_value)

for (sample_idx, token_str, token_idx), activations in tqdm(token_to_feature_activations.items()):
    for feature_idx, activation_value in enumerate(activations):
        if activation_value != 0:
            feature_to_tokens[feature_idx].append((token_str, sample_idx, token_idx, activation_value))

# Ordina i token per ogni feature in base al valore di attivazione (decrescente)
for feature_idx in feature_to_tokens:
    feature_to_tokens[feature_idx] = sorted(
        feature_to_tokens[feature_idx],
        key=lambda x: abs(x[3]),  # ordina per valore assoluto dell'attivazione
        reverse=True
    )


# Ora feature_to_tokens[feature_idx] contiene l'insieme dei token stringa attivati per ogni feature
# Esempio: mostra i primi 5 feature e i loro token associati
for feature_idx in list(feature_to_tokens.keys())[:10]:
    print(f"Feature {feature_idx}: {list(feature_to_tokens[feature_idx])[:10]}")

In [ ]:
for feature_idx in list(feature_to_tokens.keys())[:10]:
    print(f"Feature {feature_idx}: {list(feature_to_tokens[feature_idx])[:50]}")

In [ ]:
import json

# Funzione per serializzare i dati in formato richiesto
def serialize_feature_to_tokens(feature_to_tokens):
    result = {}
    for feature_idx, tokens in feature_to_tokens.items():
        result[str(feature_idx)] = [
            {
                "str": token_str,
                "sample_id": int(sample_idx),
                "token_id": int(token_idx),
                "activation": float(activation_value)
            }
            for token_str, sample_idx, token_idx, activation_value in tokens
        ]
    return result

serialized = serialize_feature_to_tokens(feature_to_tokens)

filename = f"./sae_results/feature_to_tokens_hidden{hidden_dim}_layer{layer}_beta{beta}_lr{lr}.json"
with open(filename, "w", encoding="utf-8") as f:
    json.dump(serialized, f, ensure_ascii=False, indent=2)
print(f"feature_to_tokens salvato in {filename}")

In [ ]:
import json

output = []
num_samples, num_tokens, hidden_dim = feature_activations.shape

for sample_idx in tqdm(range(num_samples)):
    tokens_json = []
    # Primo token: CLS
    nonzero = feature_activations[sample_idx, 0].nonzero().squeeze().tolist()
    if isinstance(nonzero, int):
        nonzero = [nonzero]
    activations = [
        (int(i), float(feature_activations[sample_idx, 0, i].item()))
        for i in nonzero
        if feature_activations[sample_idx, 0, i] != 0
    ]
    tokens_json.append({
        "token_idx": 0,
        "token_str": "CLS",
        "activations": activations
    })
    # print({
    #     "token_idx": 0,
    #     "token_str": "CLS",
    #     "activations": activations
    # })
    # Token successivi
    for token_idx, token in enumerate(tokens_str[sample_idx]):
        fa_idx = token_idx + 1
        if fa_idx >= num_tokens:
            break
        if token == "<pad>":
            nonzero = feature_activations[sample_idx, fa_idx].nonzero().squeeze().tolist()
            if isinstance(nonzero, int):
                nonzero = [nonzero]
            activations = [
                (int(i), float(feature_activations[sample_idx, fa_idx, i].item()))
                for i in nonzero
                if feature_activations[sample_idx, fa_idx, i] != 0
            ]
            tokens_json.append({
                "token_idx": fa_idx,
                "token_str": "</s>",
                "activations": activations
            })
            break  # Stop at first <pad>
        else:
            nonzero = feature_activations[sample_idx, fa_idx].nonzero().squeeze().tolist()
            if isinstance(nonzero, int):
                nonzero = [nonzero]
            activations = [
                (int(i), float(feature_activations[sample_idx, fa_idx, i].item()))
                for i in nonzero
                if feature_activations[sample_idx, fa_idx, i] != 0
            ]
            tokens_json.append({
                "token_idx": fa_idx,
                "token_str": token,
                "activations": activations
            })
    output.append({
        "sample_index": sample_idx,
        "tokens": tokens_json
    })

# print(json.dumps(output[0], ensure_ascii=False, indent=2))

with open("tokens_with_activations_sparse.json", "w") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"Output JSON creato con {len(output)} samples.")